# K-FAC Natural Gradient Descent on Continual Permuted MNIST
#
K-FAC (Kronecker-Factored Approximate Curvature) — efficient NGD.
vs **CBP** (Continual Backpropagation) baseline.
#
**Benchmark**: Online Permuted MNIST (CBP paper setup).  
Network: FC 784 → 2000 → 2000 → 2000 → 10, ReLU.  
800 tasks × 60 000 examples/task (full MNIST train), batch size 32.

## 1. Imports & Setup

In [1]:
import os, sys, json, time, pickle, copy, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.size'] = 11

_LOP_ROOT = "/kaggle/input/datasets/mlinh776/lop-src"
if _LOP_ROOT not in sys.path:
    sys.path.insert(0, _LOP_ROOT)

from lop.nets.deep_ffnn import DeepFFNN
from lop.algos.bp import Backprop
from lop.utils.miscellaneous import nll_accuracy as accuracy
from lop.utils.miscellaneous import compute_matrix_rank_summaries

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Metrics

In [2]:
NUM_HIDDEN_LAYERS = 5
NUM_FEATURES      = 2000
INPUT_SIZE        = 784
CLASSES_PER_TASK  = 10

@torch.no_grad()
def compute_avg_weight_magnitude(net):
    n, s = 0, 0.0
    for p in net.parameters():
        n += p.numel()
        s += torch.sum(torch.abs(p)).item()
    return s / n if n > 0 else 0.0

@torch.no_grad()
def compute_dead_neurons(net, probe_x):
    """Count dead neurons (zero activation across all probe samples) per hidden layer."""
    net.eval()
    _, hidden_acts = net.predict(probe_x)
    dead_per_layer = []
    for act in hidden_acts:
        dead_per_layer.append((act.abs().sum(dim=0) == 0).sum().item())
    return dead_per_layer

def compute_stable_rank(sv):
    if len(sv) == 0: return 0
    sorted_sv = np.flip(np.sort(sv))
    cumsum = np.cumsum(sorted_sv) / np.sum(sv)
    return int(np.sum(cumsum < 0.99) + 1)

def compute_effective_rank(singular_values):
    if len(singular_values) == 0: return 0.0
    norm_sv = singular_values / np.sum(np.abs(singular_values))
    entropy = 0.0
    for p in norm_sv:
        if p > 0.0:
            entropy -= p * np.log(p)
    return float(np.e ** entropy)

print("✓ Metrics defined (DeepFFNN / Permuted MNIST)")

✓ Metrics defined (DeepFFNN / Permuted MNIST)


## 3. MNIST Data Loading  (mirrors `load_mnist.py` from CBP repo)

In [3]:
MNIST_CACHE = "data/mnist_"

def _build_mnist_cache(cache_path: str = MNIST_CACHE):
    os.makedirs(os.path.dirname(cache_path) if os.path.dirname(cache_path) else ".", exist_ok=True)
    tfm = transforms.Compose([transforms.ToTensor()])
    train_ds = torchvision.datasets.MNIST(root="data", train=True,  transform=tfm, download=True)
    test_ds  = torchvision.datasets.MNIST(root="data", train=False, transform=tfm, download=True)

    def _load_all(ds):
        loader = DataLoader(ds, batch_size=len(ds), shuffle=True)
        imgs, labels = next(iter(loader))
        return imgs.flatten(start_dim=1), labels   # (N, 784), (N,)

    print("  Loading train …", end=" ")
    x, y = _load_all(train_ds)
    print("done.  Loading test …", end=" ")
    x_test, y_test = _load_all(test_ds)
    print("done.")

    with open(cache_path, 'wb+') as f:
        pickle.dump([x, y, x_test, y_test], f)
    print(f"  Cached at '{cache_path}'")
    return x, y, x_test, y_test


def get_mnist(cache_path: str = MNIST_CACHE):
    if os.path.isfile(cache_path):
        with open(cache_path, 'rb+') as f:
            x, y, x_test, y_test = pickle.load(f)
    else:
        print(f"Cache '{cache_path}' not found — building …")
        x, y, x_test, y_test = _build_mnist_cache(cache_path)
    return x, y, x_test, y_test


os.makedirs("data", exist_ok=True)
print("Loading MNIST …")
x_mnist, y_mnist, x_test_mnist, y_test_mnist = get_mnist()
print(f"  Train : {x_mnist.shape}  labels: {y_mnist.shape}")
print(f"  Test  : {x_test_mnist.shape}  labels: {y_test_mnist.shape}")

IMAGES_PER_CLASS  = 6000
EXAMPLES_PER_TASK = IMAGES_PER_CLASS * CLASSES_PER_TASK   # 60 000
print(f"\n  examples_per_task (change_after) = {EXAMPLES_PER_TASK}")
print("✓ MNIST ready  (CBP paper format)")

Loading MNIST …
Cache 'data/mnist_' not found — building …


  0%|          | 0.00/9.91M [00:00<?, ?B/s]

  3%|▎         | 262k/9.91M [00:00<00:03, 2.60MB/s]

 47%|████▋     | 4.69M/9.91M [00:00<00:00, 26.4MB/s]

100%|██████████| 9.91M/9.91M [00:00<00:00, 42.2MB/s]

...

  0%|          | 0.00/4.54k [00:00<?, ?B/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 17.5MB/s]

  Loading train … 

done.  Loading test … 

done.
  Cached at 'data/mnist_'
  Train : torch.Size([60000, 784])  labels: torch.Size([60000])
  Test  : torch.Size([10000, 784])  labels: torch.Size([10000])

  examples_per_task (change_after) = 60000
✓ MNIST ready  (CBP paper format)


## 4. K-FAC Natural Gradient Descent Optimizer
#
Kronecker-Factored Approximate Curvature (K-FAC) — efficient NGD.
Approximates Fisher as F ≈ A ⊗ G per layer.

In [4]:
from torch.optim import Optimizer as _Optimizer

class KFACOptimizer(_Optimizer):
    """
    K-FAC (Kronecker-Factored Approximate Curvature) Optimizer.

    Approximates the Fisher Information Matrix for each layer as:
        F ≈ A ⊗ G
    where A = input covariance, G = output gradient covariance.

    Supports: nn.Linear, nn.Conv2d. Other layers fall back to SGD.
    BatchNorm parameters are always updated with plain SGD.

    For Conv2d layers, inputs are unfolded (im2col) to create a 2D
    representation suitable for Kronecker factorization.
    """

    def __init__(self, model, lr=0.01, damping=1e-3, weight_decay=0.0,
                 T_inv=100, alpha=0.95, max_grad_norm=1.0,
                 adapt_damping=True, damping_adaptation_interval=5,
                 damping_adaptation_decay=0.99, min_damping=1e-4,
                 damping_decrease_rho_threshold=0.85,
                 damping_increase_rho_threshold=0.35):
        """
        Args:
            model: nn.Module to optimize.
            lr: Learning rate.
            damping: Initial Tikhonov damping λ. sqrt(λ) added to each factor.
            weight_decay: L2 regularization coefficient.
            T_inv: Frequency (in steps) to recompute matrix inverses.
            alpha: Exponential moving average coefficient for A and G.
            max_grad_norm: Maximum norm for natural gradient clipping.
            adapt_damping: If True, use Levenberg-Marquardt adaptive damping.
            damping_adaptation_interval: How often (in steps) to adapt damping.
            damping_adaptation_decay: Decay factor for omega computation.
            min_damping: Minimum allowed damping value.
            damping_decrease_rho_threshold: rho above this → decrease damping.
            damping_increase_rho_threshold: rho below this → increase damping.
        """
        self.model = model
        self.damping = damping
        self.weight_decay = weight_decay
        self.T_inv = T_inv
        self.alpha = alpha
        self.max_grad_norm = max_grad_norm
        self.steps = 0

        # Adaptive damping (Levenberg-Marquardt) parameters
        self.adapt_damping = adapt_damping
        self._damping_interval = damping_adaptation_interval
        self._damping_decay = damping_adaptation_decay
        self._omega = damping_adaptation_decay ** damping_adaptation_interval
        self._min_damping = min_damping
        self._rho_decrease = damping_decrease_rho_threshold
        self._rho_increase = damping_increase_rho_threshold
        self._prev_loss = float('nan')
        self._qmodel_change = float('nan')
        self._rho = float('nan')

        # Storage for Kronecker factors and their inverses
        self._modules_tracked = {}  # name -> module
        self._stats = {}            # name -> {'A': Tensor, 'G': Tensor}
        self._inv = {}              # name -> {'A_inv': Tensor, 'G_inv': Tensor}
        self._hooks = []

        defaults = dict(lr=lr)
        super().__init__(model.parameters(), defaults)

        self._register_hooks()
        print(f"  K-FAC tracking {len(self._modules_tracked)} layers")
        if adapt_damping:
            print(f"  Adaptive damping ON (interval={damping_adaptation_interval}, "
                  f"omega={self._omega:.4f}, rho_thresholds=[{damping_increase_rho_threshold}, {damping_decrease_rho_threshold}])")

    def _register_hooks(self):
        """Register forward/backward hooks on Conv2d and Linear layers."""
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Linear, nn.Conv2d)):
                self._modules_tracked[name] = module
                # Only initialize stats if not already present (preserve accumulated A/G)
                if name not in self._stats:
                    self._stats[name] = {'A': None, 'G': None}
                h1 = module.register_forward_hook(self._forward_hook(name, module))
                h2 = module.register_full_backward_hook(self._backward_hook(name, module))
                self._hooks.append(h1)
                self._hooks.append(h2)

    def _forward_hook(self, name, module):
        """Capture input activations → compute A = E[x x^T]."""
        def hook(mod, inp, out):
            if not mod.training:
                return
            with torch.no_grad():
                x = inp[0].detach()
                if isinstance(mod, nn.Conv2d):
                    x = F.unfold(x, mod.kernel_size, dilation=mod.dilation,
                                 padding=mod.padding, stride=mod.stride)
                    x = x.permute(0, 2, 1).reshape(-1, x.size(1))
                elif isinstance(mod, nn.Linear):
                    if x.dim() > 2:
                        x = x.reshape(-1, x.size(-1))
                if mod.bias is not None:
                    ones = torch.ones(x.size(0), 1, device=x.device)
                    x = torch.cat([x, ones], dim=1)
                n = x.size(0)
                cov_a = torch.matmul(x.t(), x) / n
                if self._stats[name]['A'] is None:
                    self._stats[name]['A'] = cov_a
                else:
                    self._stats[name]['A'].mul_(self.alpha).add_(cov_a, alpha=1 - self.alpha)
        return hook

    def _backward_hook(self, name, module):
        """Capture output gradients → compute G = E[g g^T]."""
        def hook(mod, grad_input, grad_output):
            if not mod.training:
                return
            with torch.no_grad():
                g = grad_output[0].detach()
                if isinstance(mod, nn.Conv2d):
                    g = g.permute(0, 2, 3, 1).reshape(-1, g.size(1))
                elif isinstance(mod, nn.Linear):
                    if g.dim() > 2:
                        g = g.reshape(-1, g.size(-1))
                n = g.size(0)
                cov_g = torch.matmul(g.t(), g) / n
                if self._stats[name]['G'] is None:
                    self._stats[name]['G'] = cov_g
                else:
                    self._stats[name]['G'].mul_(self.alpha).add_(cov_g, alpha=1 - self.alpha)
        return hook

    @torch.no_grad()
    def _invert_factors(self):
        """Invert A and G with damping. Called every T_inv steps."""
        sqrt_damping = self.damping ** 0.5
        for name in self._stats:
            A = self._stats[name]['A']
            G = self._stats[name]['G']
            if A is None or G is None:
                continue
            try:
                A_d = A + sqrt_damping * torch.eye(A.size(0), device=A.device)
                G_d = G + sqrt_damping * torch.eye(G.size(0), device=G.device)
                self._inv[name] = {
                    'A_inv': torch.linalg.inv(A_d),
                    'G_inv': torch.linalg.inv(G_d)
                }
            except RuntimeError:
                pass  # Keep previous inverse if singular

    def reset_stats(self):
        """Reset running statistics (call at task boundaries)."""
        for name in self._stats:
            self._stats[name] = {'A': None, 'G': None}
        self._inv.clear()
        self.steps = 0
        self._prev_loss = float('nan')
        self._qmodel_change = float('nan')

    def _compute_nat_grads(self):
        """Compute natural gradients for all tracked layers.
        Returns: updates dict, and (grad_dot_natgrad) for qmodel."""
        updates = {}  # name -> (nat_grad_w, nat_grad_b or None)
        grad_dot_natgrad = 0.0  # for qmodel_change: sum of grad · nat_grad

        for name, module in self._modules_tracked.items():
            if module.weight.grad is None:
                continue
            grad_w = module.weight.grad

            if name in self._inv:
                A_inv = self._inv[name]['A_inv']
                G_inv = self._inv[name]['G_inv']

                if isinstance(module, nn.Conv2d):
                    c_out = grad_w.size(0)
                    grad_2d = grad_w.reshape(c_out, -1)
                    has_bias = module.bias is not None and module.bias.grad is not None
                    if has_bias:
                        grad_2d = torch.cat([grad_2d, module.bias.grad.unsqueeze(1)], dim=1)
                    nat_grad = torch.matmul(G_inv, torch.matmul(grad_2d, A_inv))
                    if has_bias:
                        nat_grad_w = nat_grad[:, :-1].reshape_as(module.weight)
                        nat_grad_b = nat_grad[:, -1]
                    else:
                        nat_grad_w = nat_grad.reshape_as(module.weight)
                        nat_grad_b = None

                elif isinstance(module, nn.Linear):
                    has_bias = module.bias is not None and module.bias.grad is not None
                    if has_bias:
                        grad_2d = torch.cat([grad_w, module.bias.grad.unsqueeze(1)], dim=1)
                    else:
                        grad_2d = grad_w
                    nat_grad = torch.matmul(G_inv, torch.matmul(grad_2d, A_inv))
                    if has_bias:
                        nat_grad_w = nat_grad[:, :-1]
                        nat_grad_b = nat_grad[:, -1]
                    else:
                        nat_grad_w = nat_grad
                        nat_grad_b = None
                else:
                    continue

                # grad · nat_grad for qmodel (before weight decay)
                grad_dot_natgrad += (grad_w * nat_grad_w).sum().item()
                if has_bias and nat_grad_b is not None:
                    grad_dot_natgrad += (module.bias.grad * nat_grad_b).sum().item()

                if self.weight_decay > 0:
                    nat_grad_w = nat_grad_w + self.weight_decay * module.weight
                    if nat_grad_b is not None and module.bias is not None:
                        nat_grad_b = nat_grad_b + self.weight_decay * module.bias

                updates[name] = (nat_grad_w, nat_grad_b)
            else:
                # No inverse available yet → plain SGD fallback
                ngw = grad_w.clone()
                grad_dot_natgrad += (grad_w * ngw).sum().item()
                if self.weight_decay > 0:
                    ngw = ngw + self.weight_decay * module.weight
                has_bias = module.bias is not None and module.bias.grad is not None
                ngb = module.bias.grad.clone() if has_bias else None
                if has_bias and ngb is not None:
                    grad_dot_natgrad += (module.bias.grad * ngb).sum().item()
                updates[name] = (ngw, ngb)

        return updates, grad_dot_natgrad

    def _adapt_damping_step(self, loss_value):
        """Levenberg-Marquardt adaptive damping (K-FAC paper Section 6.5).

        Called every step. At each damping_adaptation_interval:
        1. Save current loss as prev_loss and compute qmodel_change
        2. On the NEXT interval boundary: compute rho and adjust damping
        """
        if not self.adapt_damping:
            return

        # At the step BEFORE the interval boundary: save loss + qmodel_change
        is_adaptation_time = ((self.steps + 1) % self._damping_interval == 0)
        if is_adaptation_time:
            self._prev_loss = loss_value
            # qmodel_change will be set after nat_grad computation (in step())

        # At the step AFTER the interval boundary: compute rho and adjust
        is_just_after = (self.steps % self._damping_interval == 0) and (self.steps > 0)
        if is_just_after and not math.isnan(self._prev_loss) and not math.isnan(self._qmodel_change):
            loss_change = loss_value - self._prev_loss
            rho = loss_change / (self._qmodel_change + 1e-12)
            self._rho = rho

            old_damping = self.damping
            # Decrease damping if qmodel is a good approximation
            should_decrease = (loss_change < 0 and self._qmodel_change > 0) or (rho > self._rho_decrease)
            should_increase = (rho < self._rho_increase)

            if should_decrease:
                self.damping = max(self.damping * self._omega, self._min_damping)
            elif should_increase:
                self.damping = self.damping / self._omega

            if self.damping != old_damping:
                # Force recompute inverses with new damping
                self._invert_factors()

    @torch.no_grad()
    def step(self, closure=None, loss_value=None):
        """K-FAC step with adaptive damping.

        Args:
            closure: Optional closure for loss computation.
            loss_value: Current mini-batch loss (float). Required for adaptive damping.
        """
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        # Adaptive damping: pre-step (compute rho from previous interval)
        if loss_value is not None:
            self._adapt_damping_step(loss_value)

        # Periodically recompute inverses
        if self.steps % self.T_inv == 0:
            self._invert_factors()

        lr = self.param_groups[0]['lr']

        # Phase 1: Compute natural gradients
        updates, grad_dot_natgrad = self._compute_nat_grads()

        # Save qmodel_change for adaptive damping
        # qmodel_change ≈ -lr * (grad · nat_grad) + 0.5 * lr² * (nat_grad · F · nat_grad)
        # Since nat_grad = F_inv · grad, nat_grad · F · nat_grad = nat_grad · grad
        # So: qmodel_change = -lr * (g·v) + 0.5 * lr² * (g·v) = -0.5 * lr * (g·v)
        if self.adapt_damping:
            is_adaptation_time = ((self.steps + 1) % self._damping_interval == 0)
            if is_adaptation_time:
                self._qmodel_change = -0.5 * lr * grad_dot_natgrad

        # Phase 2: Compute total natural gradient norm and clip
        total_norm_sq = 0.0
        for nat_grad_w, nat_grad_b in updates.values():
            total_norm_sq += nat_grad_w.norm().item() ** 2
            if nat_grad_b is not None:
                total_norm_sq += nat_grad_b.norm().item() ** 2
        total_norm = total_norm_sq ** 0.5

        clip_coef = 1.0
        if self.max_grad_norm > 0 and total_norm > self.max_grad_norm:
            clip_coef = self.max_grad_norm / (total_norm + 1e-6)

        # Phase 3: Apply updates
        for name, (nat_grad_w, nat_grad_b) in updates.items():
            module = self._modules_tracked[name]
            module.weight.data.add_(nat_grad_w, alpha=-lr * clip_coef)
            if nat_grad_b is not None and module.bias is not None:
                module.bias.data.add_(nat_grad_b, alpha=-lr * clip_coef)

        # SGD for non-tracked parameters (BatchNorm etc.)
        tracked_params = set()
        for mod in self._modules_tracked.values():
            tracked_params.add(id(mod.weight))
            if mod.bias is not None:
                tracked_params.add(id(mod.bias))

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None or id(p) in tracked_params:
                    continue
                if self.weight_decay > 0:
                    p.data.add_(p, alpha=-self.weight_decay * lr)
                p.data.add_(p.grad, alpha=-lr)

        self.steps += 1
        return loss

print("✓ KFACOptimizer defined (supports Conv2d + Linear + BN fallback)")

✓ KFACOptimizer defined (supports Conv2d + Linear + BN fallback)


## 5. Configs

In [5]:
SEED            = 42
SAVE_EVERY_N_TASKS = 80
TIME_LIMIT_SECONDS = 11.5 * 3600

# CBP paper constants (permuted_mnist/online_expr.py)
NUM_TASKS       = 800        # number of permutation tasks
CHANGE_AFTER    = 60_000     # examples per task (full MNIST train set)
MINI_BATCH_SIZE = 32         # paper uses 1 (online); 32 for K-FAC efficiency
STEPS_PER_TASK  = CHANGE_AFTER // MINI_BATCH_SIZE   # 1875
PROBE_SIZE      = 2000       # examples for rank / dead-neuron probe

_NGD_BASE = dict(
    lr=0.003, damping=1e-3, weight_decay=0, T_inv=100, alpha=0.95,
)

CONFIGS = {
    'ngd_kfac': {
        **_NGD_BASE,
    },
}

RUN_CBP = False

RESULTS_DIR = os.path.join('permuted_mnist_results', 'ngd_kfac')
CKPT_DIR    = os.path.join(RESULTS_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✓ Config: {NUM_TASKS} tasks × {STEPS_PER_TASK} steps/task, "
      f"batch={MINI_BATCH_SIZE}, network: {INPUT_SIZE}→{NUM_FEATURES}×{NUM_HIDDEN_LAYERS}→{CLASSES_PER_TASK}")

✓ Config: 800 tasks × 1875 steps/task, batch=32, network: 784→2000×5→10


## 6. Build Optimizer

In [6]:
def build_optimizer(config, model):
    return KFACOptimizer(
        model, lr=config['lr'],
        damping=config.get('damping', 1e-3),
        weight_decay=config.get('weight_decay', 0),
        T_inv=config.get('T_inv', 100),
        alpha=config.get('alpha', 0.95))

print("✓ build_optimizer defined")

✓ build_optimizer defined


## 7. Training Loop — K-FAC on Permuted MNIST

In [7]:
def _ckpt_path(method_name: str) -> str:
    return os.path.join(CKPT_DIR, f"ckpt_{method_name}.pt")


def run_ngd(method_name, config):
    """K-FAC Natural Gradient Descent on Permuted MNIST."""
    print(f"\n{'='*70}")
    print(f"  {method_name} — Permuted MNIST  ({NUM_TASKS} tasks)")
    print(f"{'='*70}")

    wall_clock_start = time.time()
    torch.manual_seed(SEED); np.random.seed(SEED)

    # Build network: DeepFFNN  784 → 2000 × 3 → 10
    net = DeepFFNN(
        input_size        = INPUT_SIZE,
        num_features      = NUM_FEATURES,
        num_outputs       = CLASSES_PER_TASK,
        num_hidden_layers = NUM_HIDDEN_LAYERS,
    ).to(device)

    optimizer = build_optimizer(config, net)
    loss_fn   = nn.CrossEntropyLoss()

    # Metric tensors  (mirrors online_expr.py)
    total_iters       = NUM_TASKS * STEPS_PER_TASK
    accuracies        = torch.zeros(total_iters,                           dtype=torch.float)
    weight_mag_log    = torch.zeros((total_iters, NUM_HIDDEN_LAYERS + 1),  dtype=torch.float)
    ranks             = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS),        dtype=torch.float)
    effective_ranks   = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS),        dtype=torch.float)
    approximate_ranks = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS),        dtype=torch.float)
    dead_neurons      = torch.zeros((NUM_TASKS, NUM_HIDDEN_LAYERS),        dtype=torch.float)
    task_metrics      = {'task_acc': [], 'task_id': [], 'avg_weight_mag': []}

    # Checkpoint resume
    ckpt_file  = _ckpt_path(method_name)
    start_task = 0
    global_iter = 0
    if os.path.isfile(ckpt_file):
        print(f"  Loading checkpoint: {ckpt_file}")
        ckpt = torch.load(ckpt_file, map_location=device, weights_only=False)
        net.load_state_dict(ckpt['model']); optimizer.load_state_dict(ckpt['optimizer'])
        accuracies        = ckpt.get('accuracies',         accuracies)
        weight_mag_log    = ckpt.get('weight_mag_log',     weight_mag_log)
        ranks             = ckpt.get('ranks',              ranks)
        effective_ranks   = ckpt.get('effective_ranks',    effective_ranks)
        approximate_ranks = ckpt.get('approximate_ranks',  approximate_ranks)
        dead_neurons      = ckpt.get('dead_neurons',       dead_neurons)
        task_metrics      = ckpt.get('task_metrics',       task_metrics)
        start_task        = ckpt['task'] + 1
        global_iter       = start_task * STEPS_PER_TASK
        print(f"  ✓ Resumed from task {ckpt['task']}")
        del ckpt; torch.cuda.empty_cache()
    else:
        print("  (no checkpoint — training from scratch)")

    def save_checkpoint(task_idx, reason="periodic"):
        ckpt_data = {
            'task': task_idx, 'model': net.state_dict(),
            'optimizer': optimizer.state_dict(),
            'accuracies': accuracies, 'weight_mag_log': weight_mag_log,
            'ranks': ranks, 'effective_ranks': effective_ranks,
            'approximate_ranks': approximate_ranks,
            'dead_neurons': dead_neurons, 'task_metrics': task_metrics,
            'params': config,
        }
        torch.save(ckpt_data, ckpt_file)
        elapsed = time.time() - wall_clock_start
        print(f"  Checkpoint saved at task {task_idx} ({reason}) [{elapsed/3600:.1f}h elapsed]")

    # Working MNIST copies (permuted in-place each task)
    x = x_mnist.clone()   # (60000, 784) float32, CPU
    y = y_mnist.clone()   # (60000,)     int64,  CPU
    time_limit_hit = False

    # ════════════════════════════════════════════════════════════════
    #  Task loop
    # ════════════════════════════════════════════════════════════════
    for task_idx in range(start_task, NUM_TASKS):
        t0 = time.time()
        elapsed = time.time() - wall_clock_start
        if elapsed > TIME_LIMIT_SECONDS:
            print(f"\n  Time limit ({elapsed/3600:.1f}h). Saving.")
            save_checkpoint(task_idx - 1, reason="time_limit")
            time_limit_hit = True; break

        iter_start = global_iter

        # ── 1. New pixel permutation + data shuffle ───────────────────
        rng        = np.random.RandomState(task_idx + SEED * 1000)
        pixel_perm = rng.permutation(INPUT_SIZE)
        data_perm  = rng.permutation(EXAMPLES_PER_TASK)
        x          = x[:, pixel_perm]
        x, y       = x[data_perm], y[data_perm]

        # ── 2. Pre-task probe: rank + dead neurons ────────────────────
        net.eval()
        with torch.no_grad():
            probe_x = x[:PROBE_SIZE].to(device)
            _, hidden_acts = net.predict(probe_x)
            for li in range(NUM_HIDDEN_LAYERS):
                act = hidden_acts[li]
                if not torch.isfinite(act).all():
                    ranks[task_idx][li] = float('nan')
                    effective_ranks[task_idx][li] = float('nan')
                    approximate_ranks[task_idx][li] = float('nan')
                    dead_neurons[task_idx][li] = 0.0
                    continue
                r, er, apr, _ = compute_matrix_rank_summaries(act, use_scipy=True)
                ranks[task_idx][li]             = r.float()
                effective_ranks[task_idx][li]   = er.float()
                approximate_ranks[task_idx][li] = apr.float()
                dead_neurons[task_idx][li] = (act.abs().sum(dim=0) == 0).sum().item()

        if task_idx % 10 == 0:
            print(f"  Task {task_idx:4d}  approx_rank={approximate_ranks[task_idx].tolist()}"
                  f"  dead={dead_neurons[task_idx].tolist()}")

        # ── 3. Train STEPS_PER_TASK mini-batch steps ──────────────────
        net.train()

        # Remove hooks before this task's eval pass
        for h in optimizer._hooks: h.remove()
        optimizer._hooks.clear()
        optimizer._register_hooks()
        optimizer.reset_stats()

        for step in range(STEPS_PER_TASK):
            start_idx = (step * MINI_BATCH_SIZE) % EXAMPLES_PER_TASK
            batch_x   = x[start_idx: start_idx + MINI_BATCH_SIZE].to(device)
            batch_y   = y[start_idx: start_idx + MINI_BATCH_SIZE].to(device)

            net.train()
            optimizer.zero_grad()
            logits = net.predict(batch_x)[0]
            loss   = loss_fn(logits, batch_y)
            loss.backward()
            optimizer.step(loss_value=loss.item())
            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                out_sm = F.softmax(net.predict(batch_x)[0], dim=1)
                accuracies[global_iter] = accuracy(out_sm, batch_y).cpu()

                for l_idx, layer_idx in enumerate(net.layers_to_log):
                    if l_idx < weight_mag_log.shape[1]:
                        weight_mag_log[global_iter][l_idx] = (
                            net.layers[layer_idx].weight.data.abs().mean())

            global_iter += 1

        # ── 4. Per-task summary ────────────────────────────────────────
        task_acc = accuracies[iter_start:global_iter].mean().item()
        task_metrics['task_acc'].append(task_acc)
        task_metrics['task_id'].append(task_idx)
        task_metrics['avg_weight_mag'].append(
            weight_mag_log[iter_start:global_iter].mean(dim=0).tolist())

        et = time.time() - t0
        print(f"  [{method_name}] Task {task_idx:4d}/{NUM_TASKS}  "
              f"acc={task_acc:.4f}  λ={optimizer.damping:.2e}  {et:.1f}s")

        if (task_idx + 1) % SAVE_EVERY_N_TASKS == 0 or task_idx == NUM_TASKS - 1:
            save_checkpoint(task_idx, reason="periodic" if task_idx < NUM_TASKS - 1 else "completed")

    # ── Save results ───────────────────────────────────────────────────
    result_file = os.path.join(RESULTS_DIR, f'{method_name}_results.pt')
    torch.save({
        'accuracies':        accuracies.cpu(),
        'weight_mag_log':    weight_mag_log.cpu(),
        'ranks':             ranks.cpu(),
        'effective_ranks':   effective_ranks.cpu(),
        'approximate_ranks': approximate_ranks.cpu(),
        'dead_neurons':      dead_neurons.cpu(),
        'task_metrics':      task_metrics,
    }, result_file)
    print(f"  ✓ Results saved to {result_file}")
    return accuracies, weight_mag_log, ranks, effective_ranks, approximate_ranks, dead_neurons, task_metrics

print("✓ run_ngd training loop defined (Permuted MNIST)")

✓ run_ngd training loop defined (Permuted MNIST)


## 8. Run Experiments

In [8]:
print("\n" + "▶" * 10 + " NGD (K-FAC) on Permuted MNIST " + "◀" * 10)
cfg = CONFIGS['ngd_kfac']
ngd_acc, ngd_wmag, ngd_ranks, ngd_er, ngd_apr, ngd_dead, ngd_task = run_ngd('ngd_kfac', cfg)


▶▶▶▶▶▶▶▶▶▶ NGD (K-FAC) on Permuted MNIST ◀◀◀◀◀◀◀◀◀◀

  ngd_kfac — Permuted MNIST  (800 tasks)


  K-FAC tracking 6 layers
  Adaptive damping ON (interval=5, omega=0.9510, rho_thresholds=[0.35, 0.85])
  (no checkpoint — training from scratch)


  Task    0  approx_rank=[687.0, 696.0, 668.0, 597.0, 520.0]  dead=[0.0, 7.0, 13.0, 34.0, 64.0]


sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


  [ngd_kfac] Task    0/800  acc=0.5525  λ=6.87e-01  43.5s


...

  [ngd_kfac] Task   69/800  acc=0.1233  λ=7.49e+03  44.3s


  Task   70  approx_rank=[695.0, 696.0, 634.0, 549.0, 473.0]  dead=[0.0, 4.0, 13.0, 39.0, 79.0]


  [ngd_kfac] Task   70/800  acc=0.2264  λ=1.50e+03  44.3s


  [ngd_kfac] Task   71/800  acc=0.1335  λ=8.21e+02  44.3s


  [ngd_kfac] Task   72/800  acc=0.1963  λ=4.49e+02  44.4s


## 9. Results Plots

In [ ]:
TASK_WINDOW = 50   # bin size for smoothing

def smooth(arr, w=TASK_WINDOW):
    n = len(arr) // w
    return np.array([arr[i*w:(i+1)*w].mean() for i in range(n)])

# Per-task accuracy (mean over STEPS_PER_TASK steps)
task_acc_arr = np.array(ngd_task['task_acc'])
x_tasks = np.arange(NUM_TASKS)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('K-FAC NGD — Continual Permuted MNIST', fontsize=14, fontweight='bold')

# Accuracy
ax = axes[0, 0]
ax.plot(x_tasks, task_acc_arr * 100, 'C0-', lw=1.2, alpha=0.6, label='per-task')
ax.plot(smooth(task_acc_arr) * 100, 'C0-', lw=2.5, label=f'{TASK_WINDOW}-task avg')
ax.set_xlabel('Task'); ax.set_ylabel('Accuracy (%)'); ax.set_title('Online Train Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)

# Approximate rank (layer 0)
ax = axes[0, 1]
for li in range(NUM_HIDDEN_LAYERS):
    ax.plot(ngd_apr[:, li].numpy(), label=f'Layer {li+1}')
ax.set_xlabel('Task'); ax.set_ylabel('Approximate Rank'); ax.set_title('Approximate Rank per Hidden Layer')
ax.legend(); ax.grid(True, alpha=0.3)

# Effective rank (layer 0)
ax = axes[0, 2]
for li in range(NUM_HIDDEN_LAYERS):
    ax.plot(ngd_er[:, li].numpy(), label=f'Layer {li+1}')
ax.set_xlabel('Task'); ax.set_ylabel('Effective Rank'); ax.set_title('Effective Rank per Hidden Layer')
ax.legend(); ax.grid(True, alpha=0.3)

# Dead neurons
ax = axes[1, 0]
for li in range(NUM_HIDDEN_LAYERS):
    ax.plot(ngd_dead[:, li].numpy(), label=f'Layer {li+1}')
ax.set_xlabel('Task'); ax.set_ylabel('Dead Neurons (count)'); ax.set_title('Dead Neurons per Hidden Layer')
ax.legend(); ax.grid(True, alpha=0.3)

# Weight magnitude (all layers)
avg_wmag = ngd_wmag.mean(dim=1).numpy()
ax = axes[1, 1]
ax.plot(x_tasks, [ngd_task['avg_weight_mag'][t][0] if len(ngd_task['avg_weight_mag']) > t else 0
                   for t in range(NUM_TASKS)], 'C0-', lw=1.5)
ax.set_xlabel('Task'); ax.set_ylabel('Avg |W|'); ax.set_title('Weight Magnitude (Layer 1)')
ax.grid(True, alpha=0.3)

# Summary text
axes[1, 2].axis('off')
n_final = min(100, len(task_acc_arr))
final_acc = task_acc_arr[-n_final:].mean() * 100
final_dead = ngd_dead[-n_final:].mean(dim=0).tolist()
final_apr  = ngd_apr[-n_final:].mean(dim=0).tolist()
summary = [f'Final {n_final}-task metrics:', '',
           f'Avg Accuracy:  {final_acc:.2f}%',
           f'Dead neurons:  {[f"{v:.0f}" for v in final_dead]}',
           f'Approx rank:   {[f"{v:.1f}" for v in final_apr]}']
axes[1, 2].text(0.1, 0.5, '\n'.join(summary), transform=axes[1, 2].transAxes,
                fontsize=12, verticalalignment='center', family='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plot_file = os.path.join(RESULTS_DIR, 'ngd_kfac_permuted_mnist.png')
plt.savefig(plot_file, dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Plot saved to {plot_file}")

## 10. Rank Evolution Plot

In [ ]:
fig2, axes2 = plt.subplots(1, NUM_HIDDEN_LAYERS, figsize=(5 * NUM_HIDDEN_LAYERS, 4), sharey=False)
fig2.suptitle('Rank Evolution per Hidden Layer — K-FAC NGD (Permuted MNIST)', fontsize=13, fontweight='bold')
LAYER_NAMES = [f'Hidden {i+1} ({NUM_FEATURES})' for i in range(NUM_HIDDEN_LAYERS)]

for li in range(NUM_HIDDEN_LAYERS):
    ax = axes2[li]
    ax.plot(ngd_ranks[:, li].numpy(),          label='rank',          color='C0', lw=1.5)
    ax.plot(ngd_er[:, li].numpy(),             label='eff. rank',     color='C1', lw=1.5)
    ax.plot(ngd_apr[:, li].numpy(),            label='approx rank',   color='C2', lw=1.5)
    ax.set_title(LAYER_NAMES[li])
    ax.set_xlabel('Task')
    if li == 0: ax.set_ylabel('Rank')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_file2 = os.path.join(RESULTS_DIR, 'rank_evolution.png')
plt.savefig(plot_file2, dpi=200, bbox_inches='tight'); plt.show()
print(f"✓ Rank evolution plot saved to {plot_file2}")

## 11. Dead Neurons Plot

In [ ]:
fig3, axes3 = plt.subplots(1, NUM_HIDDEN_LAYERS, figsize=(5 * NUM_HIDDEN_LAYERS, 4), sharey=False)
fig3.suptitle('Dead Neurons per Hidden Layer — K-FAC NGD (Permuted MNIST)', fontsize=13, fontweight='bold')

for li in range(NUM_HIDDEN_LAYERS):
    ax = axes3[li]
    ax.plot(ngd_dead[:, li].numpy(), color='C3', lw=1.5)
    ax.set_title(LAYER_NAMES[li])
    ax.set_xlabel('Task')
    if li == 0: ax.set_ylabel('Dead Neurons (count)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_file3 = os.path.join(RESULTS_DIR, 'dead_neurons.png')
plt.savefig(plot_file3, dpi=200, bbox_inches='tight'); plt.show()
print(f"✓ Dead neurons plot saved to {plot_file3}")

## 12. Sumarry

In [ ]:
n_final = min(100, len(ngd_task['task_acc']))
final_acc   = np.array(ngd_task['task_acc'][-n_final:]).mean() * 100
final_dead  = ngd_dead[-n_final:].mean(dim=0).tolist()
final_rank  = ngd_ranks[-n_final:].mean(dim=0).tolist()
final_apr   = ngd_apr[-n_final:].mean(dim=0).tolist()

print(f"\n{'='*60}")
print(f"  K-FAC NGD — Permuted MNIST — Final {n_final}-task avg:")
print(f"  Accuracy    : {final_acc:.2f}%")
print(f"  Dead neurons: {[f'{v:.0f}' for v in final_dead]}")
print(f"  Rank        : {[f'{v:.1f}' for v in final_rank]}")
print(f"  Approx rank : {[f'{v:.1f}' for v in final_apr]}")
print(f"{'='*60}")